# Automated Marine Debris Detection using Deep Learning

## 1. Introduction and Problem Statement

## 2. Data Sources

## 3. Exploratory Data Analysis (EDA)

### 3.1 Dataset Overview

### 3.2 Class Distribution

### 3.3 Image and Annotation Visualisation

### 3.4 Preprocessing and Data Splitting

## 4. Models and Methods

### 4.1 Faster R-CNN

### 4.2 Mask R-CNN

In [ ]:
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn_v2,
    MaskRCNN_ResNet50_FPN_V2_Weights
)
from torchvision.models.detection.faster_rcnn import (
    FastRCNNPredictor
)
from torchvision.models.detection.mask_rcnn import (
    MaskRCNNPredictor
)
import torch


def create_model(num_classes):
    model = maskrcnn_resnet50_fpn_v2(
        weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    )

    in_features = model.roi_heads.box_predictor.cls_score.in_features

    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # now get the number of input features for the mask classifier
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    # and replace the mask predictor with a new one
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask,
        hidden_layer,
        num_classes
    )
    return model


## 5. Experimental Setup

### 5.1 Training Configuration

In [ ]:
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from torchvision.transforms.functional import pil_to_tensor
from pycocotools.coco import COCO

class MarineDataset(Dataset):
    def __init__(self, image_dir, annotation_file, category_ids):
        self.image_dir = image_dir
        self.coco = COCO(annotation_file)
        self.image_ids = sorted(self.coco.getImgIds())
        self.category_ids = category_ids
        # Ensure consistent class IDs across train and val
    
        self.cat_to_label = {
            cat_id: i + 1
            for i, cat_id in enumerate(category_ids)
        }

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        info = self.coco.loadImgs(image_id)[0]

        image = Image.open(
            os.path.join(self.image_dir, info["file_name"])
        ).convert("RGB")

        image = pil_to_tensor(image).float() / 255.0

        ann_ids = self.coco.getAnnIds(imgIds=[image_id])
        annotations = self.coco.loadAnns(ann_ids)

        boxes, labels, masks = [], [], []

        for ann in annotations:
            if ann.get("iscrowd", 0):
                continue
            if ann["category_id"] not in self.cat_to_label:
                continue
            if not ann.get("segmentation"):
                continue

            mask = self.coco.annToMask(ann).astype(np.uint8)

            ys, xs = np.where(mask > 0)
            if len(xs) == 0:
                continue

            x1, y1 = xs.min(), ys.min()
            x2, y2 = xs.max() + 1, ys.max() + 1

            boxes.append([x1, y1, x2, y2])
            labels.append(self.cat_to_label[ann["category_id"]])
            masks.append(mask)

        _, h, w = image.shape

        target = {
            "boxes": torch.tensor(
                boxes, dtype=torch.float32
            ).reshape(-1, 4),
            "labels": torch.tensor(
                labels, dtype=torch.int64
            ),
            "masks": torch.as_tensor(
                np.stack(masks) if masks
                else np.zeros((0, h, w), dtype=np.uint8)
            ),
            "image_id": torch.tensor(image_id)
        }

        return image, target


In [ ]:
from engine import train_one_epoch, evaluate

# train on the accelerator or on the CPU, if an accelerator is not available
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device('cpu')

# our dataset has two classes only - background and person
image_dir = 
annotation_file = 
num_classes = COCO(annotation_file).getCatIds()
# use our dataset and defined transformations
dataset = MarineDataset(image_dir, annotation_file, num_classes)
dataset_val = MarineDataset(image_dir, annotation_file, num_classes)

# # split the dataset in train and test set
# indices = torch.randperm(len(dataset)).tolist()
# dataset = torch.utils.data.Subset(dataset, indices[:-50])
# dataset_val = torch.utils.data.Subset(dataset_val, indices[-50:])

# define training and validation data loaders
data_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=torch.utils.collate_fn
)

data_loader_val = torch.utils.data.DataLoader(
    dataset_val,
    batch_size=1,
    shuffle=False,
    collate_fn=torch.utils.collate_fn
)

# get the model using our helper function
model = create_model(num_classes)

# move model to the right device
model.to(device)

# construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    weight_decay=0.0005
)

# and a learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)

# let's train it just for 2 epochs
num_epochs = 15

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()
    # evaluate on the test dataset
    evaluate(model, data_loader_val, device=device)


### 5.2 Evaluation Metrics

## 6. Results

### 6.1 Quantitative Results

### 6.2 Qualitative Results

### 6.3 Model Comparison

## 7. Discussion

### 7.1 Error Analysis and Limitations

## 8. Conclusion and Future Work

## References